In [1]:
import pandas as pd

data = pd.read_csv(
    r"D:\ragas\dataset_with_answers_gemma4_e4b_denseonlybaseline.csv"
)

data.head()

,category,question,context,ground_truth,relevant_docs,answer,retrieved_docs
0,"Korupsi, Gratifikasi, Suap",Apa yang dimaksud perbuatan korupsi yang merug...,Pasal 603: Setiap Orang yang secara melawan hu...,"Berdasarkan Pasal 603 KUHP, korupsi yang merug...",['kuhp_pasal_603'],**ANALISIS HUKUM PIDANA KORUPSI**\n\n***\n\n##...,"['uu_tipikor_pasal_2', 'uu_tipikor_pasal_32', ..."
1,"Korupsi, Gratifikasi, Suap",Bagaimana hukum penyalahgunaan jabatan untuk k...,Pasal 604: Setiap Orang yang dengan tujuan men...,"Menurut Pasal 604 KUHP, penyalahgunaan jabatan...",['kuhp_pasal_604'],"Sebagai ahli hukum pidana, saya akan menganali...","['kuhp_pasal_604', 'uu_tipikor_pasal_32', 'uu_..."
2,"Korupsi, Gratifikasi, Suap",Apa perbedaan hukuman antara pemberi dan pener...,Pasal 605 ayat (1): Dipidana dengan pidana pen...,Perbedaannya terletak pada ancaman maksimum pi...,['kuhp_pasal_605'],**ANALISIS HUKUM PIDANA TENTANG SUAP (BRIBERY)...,"['kuhp_pasal_605', 'kuhp_pasal_605', 'kuhp_pas..."
3,"Korupsi, Gratifikasi, Suap",Apakah pemberian hadiah kepada pejabat tanpa p...,Pasal 606 ayat (1): Setiap Orang yang memberik...,"Ya, tetap bisa dipidana. Berdasarkan Pasal 606...",['kuhp_pasal_606'],**ANALISIS HUKUM PIDANA TERHADAP PEMBERIAN HAD...,"['kuhp_pasal_606', 'kuhp_pasal_606', 'kuhp_pas..."
4,"Korupsi, Gratifikasi, Suap",Apa saja ancaman pidana bagi setiap orang yang...,Pasal 2 ayat (1) Undang-Undang Nomor 31 Tahun ...,Berdasarkan Pasal 2 ayat (1) UU Nomor 31 Tahun...,['uu_tipikor_pasal_2'],**ANALISIS HUKUM PIDANA KORUPSI DAN PERDAYAAN ...,"['uu_tipikor_pasal_2', 'kuhp_pasal_603', 'uu_t..."


In [2]:
"""
RAGAS Evaluation — Versi Optimasi v2
Perubahan dari v1:
  - ragas_metrics sinkron dengan RAGAS_METRIC_COLS (tidak ada mismatch)
  - answer_relevancy.strictness = 1 (kurangi LLM calls dari 3x → 1x)
  - Truncate konteks sebelum evaluasi (kurangi job verifikasi)
  - MAX_TIMEOUT diturunkan sesuai model 3b
  - context_precision & context_recall dinonaktifkan secara konsisten
"""

import ast
import json
import time
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
from ragas import evaluate, RunConfig
from ragas.metrics import faithfulness, answer_relevancy
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("[INFO] Install tqdm untuk progress bar: pip install tqdm")

c:\Users\s2\miniconda3\envs\legalrag311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\s2\AppData\Local\Temp\ipykernel_3928\1493507379.py:20: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\s2\AppData\Local\Temp\ipykernel_3928\1493507379.py:20: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [3]:
# ──────────────────────────────────────────────────────────────
# LOGGING
# ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("ragas_eval_baseline.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

In [4]:
# ──────────────────────────────────────────────────────────────
# KONFIGURASI — sesuaikan di sini saja
# ──────────────────────────────────────────────────────────────
INPUT_CSV        = data               # DataFrame atau path CSV
CHECKPOINT_CSV   = "ragas_checkpoint_baseline.csv"
OUTPUT_CSV       = "ragas_per_row_baseline.csv"
SUMMARY_CSV      = "ragas_per_category_baseline.csv"
RAGAS_JSON       = "ragas_per_category_results_baseline.json"

CATEGORY_COL     = "category"
QUESTION_COL     = "question"
ANSWER_COL       = "answer"
CONTEXTS_COL     = "retrieved_docs"
GROUND_TRUTH_COL = "ground_truth"

# ── Metrik yang aktif ─────────────────────────────────────────
# Hanya 2 metrik → ~50% lebih sedikit LLM calls vs 4 metrik
# Aktifkan context_precision/recall hanya jika benar-benar dibutuhkan
RAGAS_METRIC_COLS = [
    "faithfulness",
    "answer_relevancy",
    # "context_precision",
    # "context_recall",
]

# ── Tuning kecepatan ──────────────────────────────────────────
KATEGORI_TARGET  = []    # kosong = semua; contoh: ["Korupsi", "Pidana Umum"]
MAX_RETRIES      = 3
BASE_TIMEOUT     = 120   # detik — cukup untuk ministral-3:3b
TIMEOUT_PER_CHAR = 0.03  # tambahan per karakter konteks
MAX_TIMEOUT      = 800   # batas atas; 3b tidak butuh 800s
MAX_CTX_CHARS    = 1500  # potong tiap dokumen konteks — kurangi job verifikasi
STRICTNESS       = 1     # answer_relevancy: generate N pertanyaan balik (default=3, min=1)

In [5]:
# ──────────────────────────────────────────────────────────────
# SETUP LLM & EMBEDDING
# ──────────────────────────────────────────────────────────────
log.info("Memuat LLM dan embedding model …")

ollama_llm = ChatOllama(
    model="granite4.1:8b",
    temperature=0,
    num_ctx=4096,
    request_timeout=MAX_TIMEOUT,
)

hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

ragas_llm        = LangchainLLMWrapper(ollama_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# FIX v1: ragas_metrics harus sinkron dengan RAGAS_METRIC_COLS
# Mapping nama kolom → objek metrik
_ALL_METRICS = {
    "faithfulness"      : faithfulness,
    "answer_relevancy"  : answer_relevancy,
}
ragas_metrics = [_ALL_METRICS[col] for col in RAGAS_METRIC_COLS if col in _ALL_METRICS]

# FIX v1: kurangi LLM calls answer_relevancy dari 3 → STRICTNESS
answer_relevancy.strictness = STRICTNESS

for m in ragas_metrics:
    m.llm        = ragas_llm
    m.embeddings = ragas_embeddings

log.info(f"Metrik aktif: {RAGAS_METRIC_COLS}")
log.info(f"answer_relevancy.strictness = {STRICTNESS}")
log.info("Model siap.")

2026-06-10 10:59:05,358 [INFO] Memuat LLM dan embedding model …
2026-06-10 10:59:07,781 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-10 10:59:07,808 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-06-10 10:59:08,067 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-10 10:59:08,097 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-10 10:59:08,108 [INFO] Loading SentenceTransformer model from BAAI/bge-m3.
2026-06-10 10:59:08,372 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary R

In [6]:
# ──────────────────────────────────────────────────────────────
# HELPER
# ──────────────────────────────────────────────────────────────
def parse_list(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except Exception:
        return [str(val)]


def truncate_contexts(ctx_list: list, max_chars: int = MAX_CTX_CHARS) -> list:
    """Potong tiap dokumen konteks — mengurangi jumlah klaim yang diekstrak."""
    return [str(c)[:max_chars] for c in ctx_list]


def context_length(ctx: list) -> int:
    return sum(len(c) for c in ctx)


def dynamic_timeout(ctx: list) -> int:
    t = BASE_TIMEOUT + int(context_length(ctx) * TIMEOUT_PER_CHAR)
    return min(t, MAX_TIMEOUT)

In [7]:
# ──────────────────────────────────────────────────────────────
# LOAD DATA
# ──────────────────────────────────────────────────────────────
df = INPUT_CSV.copy() if isinstance(INPUT_CSV, pd.DataFrame) else pd.read_csv(INPUT_CSV)
log.info(f"Dataset dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")

if CATEGORY_COL not in df.columns:
    log.warning(f"Kolom '{CATEGORY_COL}' tidak ditemukan — semua baris masuk kategori 'Umum'.")
    df[CATEGORY_COL] = "Umum"

# Parse + truncate konteks sekaligus
df[CONTEXTS_COL] = df[CONTEXTS_COL].apply(
    lambda v: truncate_contexts(parse_list(v))
)

df = df.reset_index(drop=True)
df["_row_id"] = df.index

for col in RAGAS_METRIC_COLS:
    if col not in df.columns:
        df[col] = np.nan

2026-06-10 10:59:16,764 [INFO] Dataset dimuat: 200 baris, 7 kolom


In [8]:
# ──────────────────────────────────────────────────────────────
# CHECKPOINT: resume progress sebelumnya
# ──────────────────────────────────────────────────────────────
checkpoint_path = Path(CHECKPOINT_CSV)
already_done: set = set()

if checkpoint_path.exists():
    ckpt = pd.read_csv(CHECKPOINT_CSV)
    # Hanya baris yang punya setidaknya 1 metrik non-NaN
    cols_in_ckpt = [c for c in RAGAS_METRIC_COLS if c in ckpt.columns]
    if cols_in_ckpt:
        done_mask    = ckpt[cols_in_ckpt].notna().any(axis=1)
        already_done = set(ckpt.loc[done_mask, "_row_id"].tolist())
        ckpt_indexed = ckpt.set_index("_row_id")
        for row_id in already_done:
            if row_id in ckpt_indexed.index:
                for col in cols_in_ckpt:
                    df.loc[df["_row_id"] == row_id, col] = ckpt_indexed.at[row_id, col]
    log.info(f"Resume checkpoint: {len(already_done)} selesai, "
             f"{len(df) - len(already_done)} tersisa.")
else:
    log.info("Tidak ada checkpoint — mulai dari awal.")

2026-06-10 10:59:16,795 [INFO] Tidak ada checkpoint — mulai dari awal.


In [9]:
# ──────────────────────────────────────────────────────────────
# EVALUASI PER-ROW
# ──────────────────────────────────────────────────────────────
def evaluate_row(row: pd.Series) -> dict:
    ctx = row[CONTEXTS_COL]
    ctx = ctx if isinstance(ctx, list) else [str(ctx)]

    dataset = Dataset.from_list([{
        "question"    : str(row[QUESTION_COL]),
        "answer"      : str(row[ANSWER_COL]),
        "contexts"    : ctx,
        "ground_truth": str(row[GROUND_TRUTH_COL]),
    }])

    run_cfg = RunConfig(
        max_workers=1,
        timeout=dynamic_timeout(ctx),
        max_retries=1,
    )

    result = evaluate(
        dataset=dataset,
        metrics=ragas_metrics,       # FIX: pakai list yang sudah sinkron
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,
        run_config=run_cfg,
    )

    result_df = result.to_pandas()
    return {
        col: float(result_df[col].iloc[0]) if col in result_df.columns else np.nan
        for col in RAGAS_METRIC_COLS
    }


def save_checkpoint():
    cols_to_save = ["_row_id", CATEGORY_COL, QUESTION_COL,
                    ANSWER_COL, GROUND_TRUTH_COL] + RAGAS_METRIC_COLS
    existing = [c for c in cols_to_save if c in df.columns]
    df[existing].to_csv(CHECKPOINT_CSV, index=False)


# ── Filter & loop ─────────────────────────────────────────────
categories = df[CATEGORY_COL].unique()
if KATEGORI_TARGET:
    categories = [c for c in categories if c in KATEGORI_TARGET]
    log.info(f"Filter kategori: {categories}")

pending_rows = df[
    df[CATEGORY_COL].isin(categories) &
    ~df["_row_id"].isin(already_done)
]

log.info(f"Akan mengevaluasi {len(pending_rows)} baris …")
print(f"\n{'═'*60}")
print(f"  RAGAS EVALUATION — {len(pending_rows)} baris pending")
print(f"  Metrik: {', '.join(RAGAS_METRIC_COLS)}")
print(f"  Model : ministral-3:3b | strictness={STRICTNESS} | ctx_max={MAX_CTX_CHARS} chars")
print(f"{'═'*60}\n")

iterator = (
    tqdm(pending_rows.iterrows(), total=len(pending_rows), unit="row")
    if HAS_TQDM else pending_rows.iterrows()
)

success_count = 0
skip_count    = 0

for idx, row in iterator:
    row_id  = row["_row_id"]
    cat     = row[CATEGORY_COL]
    q_short = str(row[QUESTION_COL])[:60]
    scores  = None
    last_exc = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            scores = evaluate_row(row)
            break
        except Exception as exc:
            last_exc = exc
            wait = 2 ** attempt          # 2s → 4s → 8s
            log.warning(f"[Row {row_id}] Attempt {attempt}/{MAX_RETRIES} gagal: {exc}. "
                        f"Tunggu {wait}s …")
            time.sleep(wait)

    if scores is not None:
        for col, val in scores.items():
            df.at[idx, col] = val
        success_count += 1
        score_str = " | ".join(
            f"{k}={v:.3f}" for k, v in scores.items() if not np.isnan(v)
        )
        log.info(f"[Row {row_id}] ✓ {cat!r} | {q_short!r} | {score_str}")
    else:
        for col in RAGAS_METRIC_COLS:
            df.at[idx, col] = np.nan
        skip_count += 1
        log.error(f"[Row {row_id}] ✗ SKIP ({MAX_RETRIES} retry habis) — {last_exc}")

    save_checkpoint()

log.info(f"Selesai: {success_count} sukses, {skip_count} di-skip.")

2026-06-10 10:59:16,822 [INFO] Akan mengevaluasi 200 baris …



════════════════════════════════════════════════════════════
  RAGAS EVALUATION — 200 baris pending
  Metrik: faithfulness, answer_relevancy
  Model : ministral-3:3b | strictness=1 | ctx_max=1500 chars
════════════════════════════════════════════════════════════



  0%|          | 0/200 [00:00<?, ?row/s]2026-06-10 10:59:32,078 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-06-10 10:59:50,464 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-06-10 11:00:16,826 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Evaluating: 100%|██████████| 2/2 [00:51<00:00, 25.59s/it]
2026-06-10 11:00:18,049 [INFO] [Row 0] ✓ 'Korupsi, Gratifikasi, Suap' | 'Apa yang dimaksud perbuatan korupsi yang merugikan keuangan ' | faithfulness=1.000 | answer_relevancy=0.792
  0%|          | 1/200 [01:01<3:23:09, 61.25s/row]2026-06-10 11:00:19,105 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-06-10 11:00:42,836 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-06-10 11:01:22,311 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Evaluating: 100%|██████████| 2/2 [01:05<00:00, 32.86s/it]
2026-06-10 11

In [10]:
# ──────────────────────────────────────────────────────────────
# RINGKASAN PER KATEGORI
# ──────────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print("  RINGKASAN PER KATEGORI")
print(f"{'═'*60}")

ragas_summary = {}
summary_rows  = []

for cat in categories:
    mask   = df[CATEGORY_COL] == cat
    sub_df = df[mask]
    n      = int(mask.sum())
    cat_scores = {}
    print(f"\n  {cat} (n={n})")
    for col in RAGAS_METRIC_COLS:
        vals     = sub_df[col].dropna()
        mean_val = float(vals.mean()) if len(vals) > 0 else float("nan")
        cat_scores[col] = round(mean_val, 4) if not np.isnan(mean_val) else None
        suffix   = f"{mean_val:.4f} (valid={len(vals)})" if not np.isnan(mean_val) else "N/A"
        print(f"    {col:<24} = {suffix}")
    ragas_summary[cat] = cat_scores
    row_dict = {
        "category" : cat,
        "n_queries": n,
        "n_valid"  : int(sub_df[RAGAS_METRIC_COLS].notna().any(axis=1).sum()),
    }
    row_dict.update(cat_scores)
    summary_rows.append(row_dict)

print(f"\n{'─'*60}")
print("  OVERALL")
overall_scores = {}
for col in RAGAS_METRIC_COLS:
    vals     = [s[col] for s in ragas_summary.values() if s[col] is not None]
    mean_val = sum(vals) / len(vals) if vals else float("nan")
    overall_scores[col] = round(mean_val, 4) if not np.isnan(mean_val) else None
    print(f"    {col:<24} = " + (f"{mean_val:.4f}" if not np.isnan(mean_val) else "N/A"))

overall_row = {
    "category" : "OVERALL",
    "n_queries": len(df),
    "n_valid"  : int(df[RAGAS_METRIC_COLS].notna().any(axis=1).sum()),
}
overall_row.update(overall_scores)
summary_rows.insert(0, overall_row)


════════════════════════════════════════════════════════════
  RINGKASAN PER KATEGORI
════════════════════════════════════════════════════════════

  Korupsi, Gratifikasi, Suap (n=15)
    faithfulness             = 0.9329 (valid=14)
    answer_relevancy         = 0.7763 (valid=15)

  Pembunuhan (n=15)
    faithfulness             = 0.9375 (valid=14)
    answer_relevancy         = 0.7377 (valid=15)

  Penganiayaan (n=15)
    faithfulness             = 0.9727 (valid=11)
    answer_relevancy         = 0.6802 (valid=13)

  Penghinaan/Pencemaran Nama Baik (n=15)
    faithfulness             = 0.7444 (valid=14)
    answer_relevancy         = 0.6796 (valid=15)

  Penyalahgunaan Jabatan (n=15)
    faithfulness             = 0.8639 (valid=15)
    answer_relevancy         = 0.6728 (valid=15)

  Penyebaran Konten Illegal (n=15)
    faithfulness             = 0.8412 (valid=15)
    answer_relevancy         = 0.6700 (valid=15)

  Produk Ilegal (Narkotika) (n=15)
    faithfulness             = 0.867

In [11]:
# ──────────────────────────────────────────────────────────────
# SIMPAN OUTPUT
# ──────────────────────────────────────────────────────────────
out_cols = ([CATEGORY_COL, QUESTION_COL, ANSWER_COL, GROUND_TRUTH_COL]
            + RAGAS_METRIC_COLS)
df[[c for c in out_cols if c in df.columns]].to_csv(OUTPUT_CSV, index=False)

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

with open(RAGAS_JSON, "w", encoding="utf-8") as f:
    json.dump(ragas_summary, f, ensure_ascii=False, indent=2)

# Hapus checkpoint jika semua baris sudah punya nilai
n_remaining = int(df[RAGAS_METRIC_COLS].isna().all(axis=1).sum())
if n_remaining == 0 and checkpoint_path.exists():
    checkpoint_path.unlink()
    log.info("Semua baris selesai — checkpoint dihapus.")
else:
    log.info(f"Checkpoint disimpan ({n_remaining} baris masih NaN).")

print(f"\n{'═'*60}")
print("  OUTPUT TERSIMPAN")
print(f"{'═'*60}")
print(f"  Detail per baris   → {OUTPUT_CSV}")
print(f"  Ringkasan kategori → {SUMMARY_CSV}")
print(f"  JSON per kategori  → {RAGAS_JSON}")
print(f"  Log evaluasi       → ragas_eval_baseline.log")
print()
print("Penjelasan metrik:")
print("  faithfulness     : jawaban didukung konteks? (0-1)")
print("  answer_relevancy : jawaban relevan ke pertanyaan? (0-1)")

2026-06-10 14:19:30,423 [INFO] Checkpoint disimpan (6 baris masih NaN).



════════════════════════════════════════════════════════════
  OUTPUT TERSIMPAN
════════════════════════════════════════════════════════════
  Detail per baris   → ragas_per_row_baseline.csv
  Ringkasan kategori → ragas_per_category_baseline.csv
  JSON per kategori  → ragas_per_category_results_baseline.json
  Log evaluasi       → ragas_eval_baseline.log

Penjelasan metrik:
  faithfulness     : jawaban didukung konteks? (0-1)
  answer_relevancy : jawaban relevan ke pertanyaan? (0-1)
